# How Top ARC-AGI-3 Agents Work

A source-level tour of Daniel Franzen's Milestone 2 agent, three public variants, and our measured V17 result. The goal is to make the moving parts legible: a game returns frames, a solver interprets the changing state, a language model proposes an action, and the harness applies it under a runtime budget.

**Evidence labels matter here.** V17's **30.65** is an official completed submission row. Other notebook scores below are claims copied from their authors' notebook text or the captured Kaggle page. They were not rerun for this explainer. Duck Lab's **31.05** is explicitly a private-set result, so it is not plotted as a public score.

This notebook uses CPU-only Python and turns internet access off in its Kaggle metadata. It does not launch an agent or submit a competition run.


## The public examples at a glance

| Notebook | What the pulled source adds or studies | Score evidence in the source | Notebook license evidence |
|---|---|---|---|
| [Franzen M2](https://www.kaggle.com/code/dfranzen/arc-agi-3-milestone-2-solution) — **Daniel Franzen** | Duck/TAAF harness, Qwen3.8 target, MTP draft model, Pennyroyal serving, and a harness patch | Kaggle page snapshot: current 33.68; best 34.30 on V1. Page claims, not rerun here. | Apache-2.0, as specified for this source in the assignment |
| [The AGI Boys](https://www.kaggle.com/code/romantamrazov/the-agi-boys-i-am-better) — **Roman Tamrazov** | Same broad serving stack; patch changes action exposure, known-no-op handling, and scheduling | No numeric score claim found in the pulled notebook | Unknown in pulled metadata and opening text |
| [Lalalia Final](https://www.kaggle.com/code/vickyfym/arc-agi-3-lalalia-final) — **Yiming Fang** | Franzen's stack plus cache-safe solved-level memory | The opening claims public leaderboard score **31.46**; not rerun here | Unknown overall; the cited memory idea is credited as CC-BY 4.0 |
| [Duck Lab T06](https://www.kaggle.com/code/yongjiehishi/arc3-duck-lab-t06) — **Yongjiehi Shi** | Scheduler and throughput ablations, including negative transfer results | **31.05** best across eight private-set draws (22.06–31.05); not a public score | Author declares CC-BY 4.0 |
| Our V17 row | Same broad TAAF / Qwen / Pennyroyal family in the V17 runner | **30.65**, official completed row `57016786` | Our row is evidence for the score only, not a license claim |

The sources overlap heavily. The useful differences are small policy changes around state reuse, action handling, and scheduling—not four unrelated model families. A companion lineage record preserves the code hashes, score provenance, and measurement method for this draft.


## 1. From observation to the next move

ARC-AGI-3 is interactive. A turn returns one or more 2D frame arrays and state metadata; the official schema describes integer cell values from 0 to 15 and grids up to 64×64. The Franzen notebook asks the Duck/TAAF harness to run the available game APIs, then hands the game state to the solver. The source patch exposes a `FrameView` with an ASCII grid, step and level numbers, shape, segmentation, previous frame, history, transitions, and valid actions. Its vision-context patch maps grid cells to a color image and can add grid differences; this is a grid-derived representation, not a raw screenshot.

In plain language, the model inspects the current frame and valid actions through a Python tool, issues one action or a short batch, then receives the refreshed frame and result. `game_over` ends a failed attempt, `level_completed` advances, and `run_complete` marks a win. The loop continues until a terminal result or a budget guard stops it. The precise underlying TAAF orchestration and baseline serializer live in the attached source bundle, so this notebook describes only interfaces visible in the pulled notebook and its patch.

The model side uses Intel's W4A16 AutoRound Qwen3.8-Flash-Next checkpoint as the target, an Albucino MTP checkpoint as the draft model, and Pennyroyal/SGLang for serving. The launcher configures three speculative steps and up to four draft tokens. The smaller draft model proposes tokens; the target model checks and accepts or replaces them. This is a decoding optimization, not the game-level action policy, and the pull alone does not establish an end-to-end speed gain.

**Action boundary.** The source notebook applies a patch to the attached harness. It changes how engine `ACTION7` is exposed as `UNDO`, controlled by `EXPOSE_UNDO`, and makes `RESET` exposure configurable. Other patched modules add no-op checks, frame/animation differences, retained functions, and priority scheduling across games. The model's move choice within a game and the harness's choice of which game runs next are separate decisions.

Sources: [ARC-AGI-3 game schema](https://github.com/arcprize/docs/blob/main/game-schema.mdx), [Franzen notebook](https://www.kaggle.com/code/dfranzen/arc-agi-3-milestone-2-solution), [Tufa Duck harness](https://www.kaggle.com/code/jeroencottaar/tufa-labs-duck-harness-june-30-milestone-winner).


In [ ]:
import json
from pathlib import Path
import ast
import base64
import html
import io
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def show_figure(fig, alt):
    image = io.BytesIO()
    fig.savefig(image, format="png", dpi=145, bbox_inches="tight")
    encoded = base64.b64encode(image.getvalue()).decode("ascii")
    label = html.escape(alt, quote=True)
    display(HTML(f'<figure><img src="data:image/png;base64,{encoded}" alt="{label}" style="max-width:100%;height:auto;"><figcaption>{label}</figcaption></figure>'))

DATA = {"score_points": [{"label": "Daniel Franzen · current public score shown on the notebook page", "score": 33.68, "kind": "notebook-reported"}, {"label": "Daniel Franzen · best score, V1, shown on the notebook page", "score": 34.3, "kind": "notebook-reported"}, {"label": "Yiming Fang · public leaderboard score claimed in the notebook opening", "score": 31.46, "kind": "notebook-reported"}, {"label": "prvsiyan · V17 official completed row", "score": 30.65, "kind": "official-row"}], "official_rows": [{"version": "V3 Stock TAAF ACTION7 Shadow", "submission_id": 55060904, "date": "2026-07-28", "status": "SubmissionStatus.COMPLETE", "official_public_score": 0.67}, {"version": "V13 earlier shadow line", "submission_id": 56432721, "date": "2026-09-21", "status": "SubmissionStatus.COMPLETE", "official_public_score": 2.95}, {"version": "V16", "submission_id": 56963315, "date": "2026-10-08", "status": "SubmissionStatus.COMPLETE", "official_public_score": 27.83}, {"version": "V17", "submission_id": 57016786, "date": "2026-10-09", "status": "SubmissionStatus.COMPLETE", "official_public_score": 30.65}], "source_stats": [{"author": "Daniel Franzen", "id": "dfranzen/arc-agi-3-milestone-2-solution", "url": "https://www.kaggle.com/code/dfranzen/arc-agi-3-milestone-2-solution", "code_cells": 11, "code_source_bytes": 536561, "code_source_lines": 10927, "functions": 70, "classes": 5}, {"author": "Roman Tamrazov", "id": "romantamrazov/the-agi-boys-i-am-better", "url": "https://www.kaggle.com/code/romantamrazov/the-agi-boys-i-am-better", "code_cells": 11, "code_source_bytes": 536686, "code_source_lines": 10927, "functions": 70, "classes": 5}, {"author": "Yiming Fang", "id": "vickyfym/arc-agi-3-lalalia-final", "url": "https://www.kaggle.com/code/vickyfym/arc-agi-3-lalalia-final", "code_cells": 11, "code_source_bytes": 545693, "code_source_lines": 11131, "functions": 80, "classes": 5}, {"author": "Yongjiehi Shi", "id": "yongjiehishi/arc3-duck-lab-t06", "url": "https://www.kaggle.com/code/yongjiehishi/arc3-duck-lab-t06", "code_cells": 12, "code_source_bytes": 555839, "code_source_lines": 10966, "functions": 71, "classes": 5}], "game_file_roots": ["/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"], "game_file_count_listed": 25, "game_file_example": "environment_files/ar25/0c556536/ar25.py"}
SCORE_POINTS = DATA["score_points"]
OFFICIAL_ROWS = DATA["official_rows"]
SOURCE_STATS = DATA["source_stats"]

print(
    f"Embedded source score points: {len(SCORE_POINTS)} | "
    f"completed official rows: {len(OFFICIAL_ROWS)} | "
    f"notebooks measured: {len(SOURCE_STATS)}"
)


## 2. Keep score types separate

Franzen's page snapshot includes both a current score and a best historical score. Lalalia's opening text states a public score. Our V17 number comes from a completed official row. These are useful reference points, but the values do not share one run date or source format. The plot labels each evidence type; it is not a controlled head-to-head test.


In [ ]:
colors = ["#c36d3f" if p["kind"] == "official-row" else "#277c78" for p in SCORE_POINTS]
fig, ax = plt.subplots(figsize=(11.2, 4.7), layout="constrained")
y = list(range(len(SCORE_POINTS)))
bars = ax.barh(y, [p["score"] for p in SCORE_POINTS], color=colors, height=0.66)
ax.set_yticks(y, [p["label"] for p in SCORE_POINTS])
ax.invert_yaxis()
ax.set_xlim(0, 40)
ax.set_xlabel("Public score as reported in source (%)")
ax.set_title("Public score references — source types are not a matched rerun", loc="left", weight="bold")
ax.xaxis.grid(True, color="#d9e1e5", linewidth=0.8)
ax.set_axisbelow(True)
for bar, point in zip(bars, SCORE_POINTS):
    ax.text(bar.get_width() + 0.35, bar.get_y() + bar.get_height()/2,
            f"{point['score']:.2f} · {'official row' if point['kind']=='official-row' else 'reported'}",
            va="center", fontsize=9)
ax.text(0, -0.17, "Franzen page snapshot: current 33.68, best V1 34.30 · Lalalia opening: 31.46 · V17 complete row: 30.65",
        transform=ax.transAxes, fontsize=8.5, color="#475569")
show_figure(fig, "Horizontal comparison of author-reported public notebook scores and the official completed V17 row, with evidence type labels.")
plt.close(fig)


## 3. Our measured history

Every point below is a completed official public-score row recorded in the lane validation evidence. The submission IDs are shown so the chart remains traceable to those rows. No local score was generated for this explainer.


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.0), layout="constrained")
xs = list(range(len(OFFICIAL_ROWS)))
ys = [row["official_public_score"] for row in OFFICIAL_ROWS]
ax.plot(xs, ys, color="#315d80", linewidth=2.2, marker="o", markersize=7)
for x, row in zip(xs, OFFICIAL_ROWS):
    ax.annotate(f"{row['version']}\n{row['official_public_score']:.2f}\n#{row['submission_id']}",
                (x, row["official_public_score"]), xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=8.5)
ax.set_xticks(xs, [r["date"] for r in OFFICIAL_ROWS])
ax.set_ylabel("Official public score (%)")
ax.set_ylim(0, 36)
ax.set_title("prvsiyan ARC-AGI-3 — completed official rows", loc="left", weight="bold")
ax.grid(axis="y", color="#d9e1e5")
show_figure(fig, "Line chart of four completed official prvsiyan ARC-AGI-3 submission scores, labeled by version, submission ID, and date.")
plt.close(fig)


## 4. What differs among the agents

| Source | Decision change or experiment | Evidence limit |
|---|---|---|
| Franzen M2 | The reference path: Duck/TAAF orchestration, a Qwen target plus MTP drafting, and action-name/harness wiring. | The notebook says the four-pass demo rerun was not completed before publication; its page score is reported evidence. |
| The AGI Boys | Adds RESET/UNDO exposure handling, a repeat guard keyed to the game state and action, and a scheduler priority rule. | Its pulled notebook does not state a numeric score. |
| Lalalia | Saves a short solved-level note and appends it only after history trimming, aiming to reuse rules without breaking the shared prompt prefix cache. | Its 31.46 is the author's public-score statement; the 94.3% cache-hit figure is also author-reported. |
| Duck Lab T06 | Measures throughput and scheduling changes; reports that action throughput gains did not transfer cleanly to the private set. | Its 31.05 and ablation values are private-set author reports, not public leaderboard scores. |
| Our V17 | A complete official score row for the TAAF-based runner. | This is one completed submission, not a repeated-run distribution. |

**A useful reading rule:** more actions per second, a better no-op policy, or a plausible priority formula is a hypothesis until the same code and evaluation set show a repeatable full-job gain.


In [ ]:
fig, ax = plt.subplots(figsize=(13.2, 5.6), layout="constrained")
ax.set_xlim(0, 13.2); ax.set_ylim(0, 5.6); ax.axis("off")
boxes = [
    (0.25, 2.55, 2.3, 1.55, "1  Observation", "FrameView: ASCII + shape\nsegmentation + valid actions"),
    (2.85, 2.55, 2.3, 1.55, "2  Duck / TAAF solver", "Model inspects state with\nPython; sees refreshed turns"),
    (5.45, 2.55, 2.3, 1.55, "3  Model serving", "Qwen3.8 target checks\n3 draft steps / up to 4 tokens"),
    (8.05, 2.55, 2.3, 1.55, "4  Action translation", "Action list / short batch\n→ engine action mapping"),
    (10.65, 2.55, 2.3, 1.55, "5  Next state", "Frame(s) + terminal flag\nwin · level · game over"),
]
palette = ["#e8f1f6", "#e9f2ed", "#f6efe6", "#f5e9e8", "#e8f1f6"]
for i, (x, y, w, h, title, detail) in enumerate(boxes):
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04,rounding_size=0.12",
                           linewidth=1.3, edgecolor="#536879", facecolor=palette[i])
    ax.add_patch(patch)
    ax.text(x+0.12, y+h-0.34, title, fontsize=10, weight="bold", color="#17324d")
    ax.text(x+0.12, y+0.38, detail, fontsize=8.7, va="center", color="#263746")
    if i < len(boxes)-1:
        ax.add_patch(FancyArrowPatch((x+w+0.04, y+h/2), (boxes[i+1][0]-0.06, y+h/2),
                                     arrowstyle="-|>", mutation_scale=13, color="#50677a", linewidth=1.5))
ax.add_patch(FancyBboxPatch((2.4, 0.55), 8.4, 0.9, boxstyle="round,pad=0.04,rounding_size=0.1",
                           linewidth=1.1, edgecolor="#7e6a43", facecolor="#fbf5e8"))
ax.text(6.6, 1.0, "Run controller: per-game limits · analyzer timeout · pass count · concurrency · soft end before the 9-hour wall cap",
        ha="center", va="center", fontsize=9.5, weight="bold", color="#574624")
ax.add_patch(FancyArrowPatch((6.6, 1.48), (6.6, 2.45), arrowstyle="-|>", mutation_scale=13,
                             color="#8b7347", linewidth=1.3))
ax.text(0.28, 5.08, "MTP speculative decoding is inside step 3; cross-game priority scheduling is a separate run-controller decision.",
        fontsize=9, color="#425466")
ax.set_title("One interactive turn in the Franzen / Duck-TAAF family", loc="left", weight="bold", fontsize=13)
show_figure(fig, "Five-step ARC-AGI-3 agent turn diagram showing observation, Duck/TAAF solver, target and draft model serving, action translation, returned state, and run controller.")
plt.close(fig)


## 5. How much code is visible in each pull?

The next chart counts source text in code cells from the four pulled notebooks. The unusually large code cells contain the harness patch and serving setup; bytes and line counts measure packaging, not agent quality. The notebook bodies are preserved in `public-pulls/`, and hashes are in `lineage.json`.


In [ ]:
labels = [s["author"] for s in SOURCE_STATS]
mib = [s["code_source_bytes"] / (1024**2) for s in SOURCE_STATS]
fig, ax = plt.subplots(figsize=(10.5, 4.6), layout="constrained")
y = list(range(len(SOURCE_STATS)))
bars = ax.barh(y, mib, color=["#315d80", "#7e9f83", "#9a7baa", "#c38a50"])
ax.set_yticks(y, labels); ax.invert_yaxis()
ax.set_xlabel("Code-cell source (MiB)")
ax.set_title("Pulled notebook code structure", loc="left", weight="bold")
for bar, s in zip(bars, SOURCE_STATS):
    ax.text(bar.get_width()+0.015, bar.get_y()+bar.get_height()/2,
            f"{s['code_source_bytes']:,} bytes · {s['code_cells']} code cells",
            va="center", fontsize=8.5)
ax.set_xlim(0, max(mib)*1.55)
ax.grid(axis="x", color="#d9e1e5"); ax.set_axisbelow(True)
show_figure(fig, "Horizontal bars comparing code-cell source bytes and code-cell counts across four pulled ARC-AGI-3 notebooks."); plt.close(fig)

print("Definitions in source code cells (simple line-based counts):")
for s in SOURCE_STATS:
    print(f"- {s['author']}: {s['code_source_lines']:,} lines, {s['functions']} functions, {s['classes']} classes")


## 6. Budget and time controls

In the pulled source, the demo path uses four passes over 25 games; with priority scheduling on, its concurrency is 100 and its per-game cap is calculated as `532 min × 100 / 110` (about 8 h 4 min). In the competition branch it disables request-log saving, uses one pass, sets concurrency to 120, and sets a 532-minute per-game cap. The source checks a 32,400-second (9-hour) wall budget and computes a soft end 600 seconds before it (8 h 50 min). The analyzer timeout is 900 seconds and server startup wait is 12 minutes. These are configured controls, not measured completion times. The initial `48*60` assignment is overwritten when priority scheduling is enabled; its inline `#7920.0` comment does not match that expression.


In [ ]:
values = [540, 530, 532, 532*100/110, 15, 12]
labels = ["Competition wall cap", "Soft end", "Competition per-game cap", "Demo per-game cap*", "Analyzer timeout", "Server startup wait"]
fig, ax = plt.subplots(figsize=(10.8, 4.8), layout="constrained")
bars = ax.barh(range(len(values)), values,
               color=["#b7c6d1", "#315d80", "#9a7baa", "#7e9f83", "#c38a50", "#a9b6a6"])
ax.set_yticks(range(len(labels)), labels); ax.invert_yaxis()
ax.set_xlim(0, 600); ax.set_xlabel("Minutes (configured limits, not measured runtimes)")
ax.set_title("Runtime controls in the pulled source", loc="left", weight="bold")
for bar, value in zip(bars, values):
    ax.text(value+5, bar.get_y()+bar.get_height()/2, f"{value:.1f} min", va="center", fontsize=8.5)
ax.grid(axis="x", color="#d9e1e5"); ax.set_axisbelow(True)
ax.text(0, -0.15, "*Demo value is 532 × 100 / 110 with four passes over 25 games; competition branch uses one pass and concurrency 120.",
        transform=ax.transAxes, fontsize=8, color="#475569")
show_figure(fig, "Horizontal bars of configured runtime limits from the Franzen notebook source, separating wall-clock, per-game, analyzer, and startup controls."); plt.close(fig)


## 7. The public game files

The competition file listing contains 25 public game Python files and 25 `metadata.json` files in `environment_files/`. For example: `environment_files/ar25/0c556536/ar25.py`. The chart reads those mounted source files when this notebook runs on Kaggle and reports each file's size and number of top-level function definitions. It never imports or executes a game file. On this local machine, `/kaggle/input` is absent, so the chart deliberately renders an availability note instead of inventing sample values.

The file-size chart describes the public game implementations; it is not a difficulty or score chart. ARC game observations are described in the official schema linked above.


In [ ]:
game_root = next((Path(p) for p in DATA["game_file_roots"] if Path(p).is_dir()), None)
fig, ax = plt.subplots(figsize=(11.2, 6.0), layout="constrained")
if game_root is None:
    ax.axis("off")
    ax.text(0.02, 0.74, "Competition game files are not mounted in this local run.",
            transform=ax.transAxes, fontsize=15, weight="bold", color="#17324d")
    ax.text(0.02, 0.57, "At Kaggle runtime this cell reads environment_files/<game>/<version>/*.py\nand plots observed file bytes and parsed function counts.",
            transform=ax.transAxes, fontsize=11, color="#425466")
    ax.text(0.02, 0.36, f"Competition listing observed: {DATA['game_file_count_listed']} public game files; example {DATA['game_file_example']}",
            transform=ax.transAxes, fontsize=10, color="#425466")
    ax.set_title("Public ARC-AGI-3 environment source files", loc="left", weight="bold")
    game_stats = []
else:
    game_stats = []
    for path in sorted(game_root.rglob("*.py")):
        if path.is_symlink():
            continue
        raw = path.read_bytes()
        try:
            tree = ast.parse(raw.decode("utf-8"))
            functions = sum(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) for node in tree.body)
        except (SyntaxError, UnicodeDecodeError):
            functions = 0
        game_stats.append({"name": path.parent.parent.name, "bytes": len(raw), "functions": functions})
    game_stats.sort(key=lambda g: g["bytes"])
    names = [f"{g['name']}  ·  {g['functions']} functions" for g in game_stats]
    ax.barh(range(len(game_stats)), [g["bytes"]/1024 for g in game_stats], color="#527e83")
    ax.set_yticks(range(len(game_stats)), names, fontsize=7)
    ax.set_xlabel("Python source size (KiB)")
    ax.set_title(f"Competition-mounted public game files · {len(game_stats)} files", loc="left", weight="bold")
    ax.grid(axis="x", color="#d9e1e5"); ax.set_axisbelow(True)
show_figure(fig, "Public ARC-AGI-3 environment source file sizes and parsed top-level function counts, or a note that competition inputs are not mounted locally."); plt.close(fig)
print(f"Mounted public game source files inspected: {len(game_stats)}")


## Credits, boundaries, and what to try next

- **Tufa Labs / Jeroen Cottaar:** Duck solver, TAAF game harness, and upstream bundle. **Daniel Franzen:** Milestone 2 notebook and harness patch. His source says the attached bundle is an unchanged copy and attributes the original harness to Tufa Labs.
- **John Pezzulli:** Pennyroyal/SGLang serving work. **Intel:** AutoRound target checkpoint. **Albucino:** MTP draft checkpoint. The Franzen source also credits Mamy Ratsimbazafy and Gabriel Olympie for SGLang patches.
- **Roman Tamrazov:** The AGI Boys variant and its patch. **Yiming Fang:** Lalalia's solved-level-memory variant; its opening credits the underlying memory idea to rellik13 and sirikilohit (CC-BY 4.0). **Yongjiehi Shi:** Duck Lab T06's ablations; its opening credits the D′ priority port to daydreamerthinks and shiiin9 and declares CC-BY 4.0.
- **ARC Prize Foundation:** competition game files and the public observation schema. Public competition code is inspected for source-size counts only; game code is not executed here.
- **prvsiyan:** V17's 30.65 is attributed to completed official submission row `57016786`, recorded in the lane validation file.

Overall licenses for The AGI Boys and Lalalia were not found in the pulled metadata or opening text, so this notebook records them as **unknown**. The cited component licenses do not automatically set the license of a whole notebook. Links above preserve the original sources; this notebook does not copy their agent code.

**What to try next:** reproduce the source-reported score claims with matched public runs; test no-op handling and solved-level memory as separate ablations; then compare full-job scores and runtimes rather than throughput alone.

Thank you to the authors for publishing work that others can inspect and learn from.
